UNIVERSIDADE DO VALE DO ITAJAÍ<br>
ESCOLA POLITÉCNICA<br>
CURSO DE ENGENHARIA DE COMPUTAÇÃO<br>
<br>
Implementação original: Sydney Matheus de Souza<br>
Reimplementação em PyTorch: Eduardo Savian<br>
Orientação: Felipe Viel<br>

# Classificação de imagens hiperespectrais com CNN 1D

Reimplementação em PyTorch do notebook `TCC3_SydneySouza_01-10-2023.ipynb`, reorganizada
como biblioteca: a lógica vive em `src/hsi/` e este notebook é só o roteiro que a executa e
mostra os resultados. O CLI `scripts/run_experiment.py` chama exatamente as mesmas funções,
então notebook e linha de comando produzem resultados idênticos.

**Execução:** local (RTX 5070) ou instância AWS — sem Google Colab e sem Google Drive.
Os datasets ficam em disco (`data/dataHSI/`, ou onde `HSI_DATA_DIR` apontar) e são obtidos
por `scripts/download_data.py`. Instruções de ambiente em `SETUP.md`.

## Decisões de implementação

- **Layout do espectro:** `(N, 1, Kb)`, *channels-first*, que é o que o `nn.Conv1d` espera —
  uma sequência de comprimento Kb com um único canal.
- **Logits, não softmax:** a última camada devolve logits e quem aplica o softmax é a
  `nn.CrossEntropyLoss` durante o treino. Na predição ele é aplicado explicitamente. Isso
  evita calcular log(softmax) duas vezes e é numericamente mais estável.
- **Rótulos como índices:** `CrossEntropyLoss` recebe o índice da classe (`int64`), então não
  há one-hot em lugar nenhum do pipeline.
- **Inicialização Xavier/Glorot** com bias zerado, em vez do Kaiming uniforme que o PyTorch usa
  por padrão. É a inicialização da implementação de referência e muda a curva de convergência.
- **Parada antecipada** em `val_loss` com `patience=5`, sem restaurar os pesos da melhor época:
  os pesos finais são os da última. O mesmo comportamento da implementação de referência.
- **`Adam` com `eps=1e-7`** (o padrão do PyTorch é `1e-8`), também para manter a comparação.

## Mapa do pacote

| módulo | responsabilidade |
|---|---|
| `hsi.config` | `ExperimentConfig` — todos os hiperparâmetros num objeto só |
| `hsi.paths` | raiz do repositório, diretório de dados e de saídas |
| `hsi.env` | versões, seleção de GPU, determinismo e sementes |
| `hsi.data` | carga dos `.mat`, tabela de pixels, remoção do background |
| `hsi.preprocess` | PCA e composição RGB |
| `hsi.segmentation` | superpixels SEEDS |
| `hsi.dataset` | split estratificado e DataLoaders |
| `hsi.parallel` | execução concorrente dos ramos independentes da preparação |
| `hsi.variants` | baseline (Souza, 2023) × nossa implementação |
| `hsi.model` | `CNN1D` |
| `hsi.train` | laço de treino e early stopping |
| `hsi.evaluate` | predição, OA/AA/kappa, matriz de confusão |
| `hsi.plots` | curvas e figuras |
| `hsi.pipeline` | encadeia tudo (`prepare_data`, `run_experiment`) |

# Initialization

## Imports e diretórios

O notebook original montava o Google Drive e usava `os.getcwd()`. Aqui a raiz do repositório
é descoberta subindo a árvore de diretórios, o que funciona igual na máquina local e numa
instância AWS. `HSI_DATA_DIR` tem precedência sobre `data/dataHSI/` (útil para um EBS/EFS).

In [ ]:
import sys
from pathlib import Path


# Localiza src/ subindo a árvore: funciona com o kernel iniciado em qualquer nível
# do repositório (na raiz, dentro de src/, etc.).
def find_src():
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / 'src' / 'hsi' / '__init__.py').exists():
            return base / 'src'
    raise RuntimeError('pacote hsi não encontrado; inicie o Jupyter dentro do repositório')


sys.path.insert(0, str(find_src()))

import numpy as np
import torch
import matplotlib.pyplot as plt

import hsi
from hsi import ExperimentConfig, data_dir, output_dir, repo_root

REPO_ROOT = repo_root()
DATA_DIR = data_dir(REPO_ROOT)
OUT_DIR = output_dir(REPO_ROOT)

print('hsi:      ', hsi.__version__)
print('REPO_ROOT:', REPO_ROOT)
print('DATA_DIR: ', DATA_DIR, '(existe)' if DATA_DIR.is_dir() else '(será criado no download)')
print('OUT_DIR:  ', OUT_DIR)

## Versões (reprodutibilidade)

Compara o ambiente com as versões usadas na validação e apenas avisa nas divergências —
sem interromper, para você julgar se a diferença importa. As mesmas versões estão pinadas
em `requirements.txt`.

O build CUDA do PyTorch **não** é pinável no `requirements.txt` (depende do índice usado na
instalação), por isso é reportado à parte: `cu129`/`cu130` para a RTX 5070.

In [ ]:
hsi.check_versions();

## GPU e determinismo

Duas armadilhas detectadas aqui em vez de estourarem no meio do treino:

- **RTX 5070 (Blackwell, `sm_120`)** só roda em build do PyTorch compilado com CUDA 12.8+.
  Um wheel cu121/cu124 instala e importa sem erro, mas falha no primeiro kernel com
  *"no kernel image is available for execution on the device"*. A verificação compara a
  compute capability da GPU com `torch.cuda.get_arch_list()` e ainda executa uma convolução
  de teste — que é a prova definitiva.
- **Instâncias AWS** têm arquiteturas diferentes (g5 = A10G `sm_86`, g6 = L4 `sm_89`,
  p4d = A100 `sm_80`, p5 = H100 `sm_90`). O mesmo notebook serve para todas.

`configure_determinism(True)` desliga `cudnn.benchmark` e TF32: com `benchmark=True` o cuDNN
mede e escolhe o kernel mais rápido por shape, e a escolha pode variar entre execuções; TF32
é determinístico, mas trunca a mantissa e dá números diferentes de FP32 puro. Troque para
`False` só ao medir desempenho.

In [ ]:
device = hsi.select_device()
hsi.configure_determinism(deterministic=True)

# Configuração do experimento

Tudo o que antes eram variáveis globais espalhadas por várias células (`dataset`,
`test_ratio`, `two_input`, `Kb`, `NUM_EPOCHS`, `BATCH_SIZE`, `SEED`, ...) agora é um objeto
só. Mude aqui e o resto do notebook acompanha.

- `dataset`: `'IP'` (Indian Pines), `'PU'` (Pavia University) ou `'SA'` (Salinas).
- `two_input`: `True` usa espectro + segmentação SEEDS; `False`, só o espectro.
- `pca_components` é derivado do dataset (30 para IP, 15 para os demais) se não for informado.

In [ ]:
config = ExperimentConfig(
    dataset='PU',
    two_input=True,
    test_ratio=0.15,
    valid_ratio=0.5,
    epochs=60,
    batch_size=32,
    seed=42,
)

print(config.summary())

# Data Prep

## Load

`load_scene` baixa os `.mat` que faltarem (chamando `scripts/download_data.py`, que valida
sha256 e conteúdo) e devolve o cubo `(H, W, B)` com o ground truth `(H, W)`.

In [ ]:
scene = hsi.load_scene(config.dataset, DATA_DIR)
print(scene)
print('Dimensões da cena:', scene.shape, '| bandas:', scene.n_bands)

## Distribuição das classes

Cada pixel é uma amostra. O rótulo 0 é background e é descartado; as classes restantes são
reindexadas para `0..C-1`. As cenas são bem desbalanceadas, e é por isso que os splits são
estratificados e que a AA (que pesa cada classe igualmente) fica sempre abaixo da OA.

In [ ]:
import pandas as pd

pixels, raw_labels = hsi.to_pixel_table(scene)
spectra, labels = hsi.remove_background(pixels, raw_labels)

print('Tabela de pixels: ', pixels.shape, '->', spectra.shape, 'após remover o background')
print('Amostras válidas: ', len(labels), f'({100 * len(labels) / len(raw_labels):.1f}% da cena)')
print('\nAmostras por classe:')
print(pd.Series(labels).value_counts().sort_index().to_string())

## PCA e segmentação

`prepare_data` executa o pipeline inteiro de preparação:

1. tabela de pixels e remoção do background;
2. **PCA** para `config.pca_components` bandas (`whiten=True`, `random_state=42`);
3. quando `two_input=True`: PCA de 3 componentes sobre a cena **inteira** (o SEEDS precisa
   da imagem completa, background incluído), composição RGB e segmentação em superpixels;
4. split estratificado em treino / validação / teste.

O id do superpixel de cada pixel é a segunda entrada da rede.

In [ ]:
data, segmentation, prep_timings = hsi.prepare_data_parallel(
    config, DATA_DIR, scene=scene, backend='sequential')

print('Espectro após PCA:', data.spectra.shape, data.spectra.dtype)
print('Classes:          ', data.n_classes)
print('Partições:        ', data.splits.sizes())
if segmentation is not None:
    print('Superpixels:      ', segmentation.n_superpixels,
          f'(alvo: {hsi.SEEDS_PARAMS[config.dataset].num_superpixels})')
print()
print(prep_timings.summary())

### Componentes principais usadas na segmentação

In [ ]:
if segmentation is not None:
    hsi.plots.plot_pca_components(segmentation.components, scene.shape)
    plt.show()

### Canais e composição RGB que alimentam o SEEDS

In [ ]:
if segmentation is not None:
    hsi.plots.plot_rgb_channels(segmentation.rgb)
    plt.show()

    plt.figure(figsize=(10, 8))
    plt.imshow(segmentation.rgb)
    plt.axis('off')
    plt.title('Composição RGB (3 primeiras componentes)')
    plt.show()

### Contornos e superpixels resultantes

In [ ]:
if segmentation is not None:
    hsi.plots.plot_segmentation_steps(hsi.contour_overlay(segmentation.rgb, segmentation))
    plt.show()

    plt.figure(figsize=(10, 8))
    plt.imshow(hsi.colorize_segments(segmentation.labels, seed=config.seed))
    plt.axis('off')
    plt.title(f'{segmentation.n_superpixels} superpixels')
    plt.show()

# Paralelização inter-etapas

Depois do PCA, os dois ramos da preparação são independentes até o ponto de fusão:

- **ramo espectral** — PCA para `config.pca_components` bandas sobre os pixels válidos;
- **ramo espacial** — PCA de 3 componentes sobre a cena inteira, composição RGB e SEEDS.

Nenhum precisa do resultado do outro; só a fusão (montar a segunda entrada da rede) precisa
dos dois. Executá-los concorrentemente troca `T_espectral + T_espacial` por
`max(T_espectral, T_espacial) + overhead`.

| backend | como executa | observação |
|---|---|---|
| `sequential` | um ramo depois do outro | linha de base da comparação |
| `thread` | `ThreadPoolExecutor` | funciona apesar do GIL: a SVD do PCA (BLAS) e o `iterate` do SEEDS (OpenCV) o liberam; não copia nada |
| `process` | `ProcessPoolExecutor` | imune ao GIL, mas paga serialização da tabela de pixels |

O resultado é **idêntico** nos três: o paralelismo muda quando cada etapa roda, não o que
ela calcula. A célula abaixo verifica exatamente isso, além de medir os tempos.

In [ ]:
import pandas as pd

linhas, referencia = [], None
for backend in hsi.BACKENDS:
    d, _, t = hsi.prepare_data_parallel(config, DATA_DIR, scene=scene, backend=backend)
    chave = (d.spectra, d.segments)
    if referencia is None:
        referencia, identico = chave, True
    else:
        identico = all(np.array_equal(a, b) for a, b in zip(referencia, chave))
    linhas.append({
        'backend': backend,
        'bloco (s)': t.wall,
        'espectral (s)': t.stages.get('espectral', float('nan')),
        'espacial (s)': t.stages.get('espacial', float('nan')),
        'soma ramos (s)': t.branch_sum,
        'overhead (s)': t.overhead,
        'idêntico': identico,
    })

tabela = pd.DataFrame(linhas).set_index('backend')
tabela['speedup'] = tabela.loc['sequential', 'bloco (s)'] / tabela['bloco (s)']
print(tabela.to_string(float_format='{:.3f}'.format))

Uma ressalva que o próprio número deixa claro: a preparação é uma fração pequena do tempo de
ponta a ponta — o treino domina. Pela lei de Amdahl, o ganho no total fica limitado ao peso
da preparação, mesmo que o bloco paralelo em si acelere bem. Os scripts
`scripts/benchmark_parallel.py` e `scripts/compare_variants.py` medem as duas coisas: o ganho
no bloco e o que ele representa no total.

## DataLoaders

O original empilhava espectro e segmentação num tensor `(N, Kb, 1, 1, 2)` só para passar os
dois pelo mesmo `train_test_split`, e depois desfazia o empilhamento com fatiamento
posicional. Aqui o **índice** é dividido uma vez e cada array é indexado por ele — mesma
partição, sem o malabarismo de shape.

O loader de treino recebe um `generator` semeado explicitamente: a ordem dos batches passa a
depender só de `config.seed`, então reexecutar apenas a célula de treino reproduz o mesmo
resultado.

In [ ]:
loaders = hsi.build_loaders(data, config, device)

# Inspeciona os tensores pelo dataset, não iterando o loader: criar um iterador
# sobre o loader de treino sortearia uma permutação e adiantaria o gerador semeado.
x, *rest = loaders['train'].dataset.tensors
print('Tensor de entrada (HSI):', tuple(x.shape), x.dtype, '— channels-first para o nn.Conv1d')
if config.two_input:
    print('Tensor de entrada (segmentação):', tuple(rest[0].shape), rest[0].dtype)
print('Batches -> treino:', len(loaders['train']),
      '| val:', len(loaders['val']), '| teste:', len(loaders['test']))

# CNN

## Modelling

Quatro convoluções 1D sem padding sobre o vetor espectral, seguidas de uma MLP com dropout:

| bloco | camadas |
|---|---|
| extração | `Conv1d(1→16, k=9)` → `Conv1d(16→32, k=3)` → `Conv1d(32→64, k=3)` → `Conv1d(64→64, k=3)`, ReLU em cada uma, e `Flatten` |
| classificação | `Linear(→128)` → `Dropout(0.5)` → `Linear(→64)` → `Dropout(0.5)` → `Linear(→classes)` |

Com `two_input=True`, o id do superpixel é concatenado ao vetor achatado antes da MLP: além
de "com o que este pixel se parece", a rede passa a saber "de que região homogênea ele veio".

O tamanho da entrada do primeiro `Linear` é inferido com um forward simbólico, evitando
calcular à mão o encolhimento da sequência pelas convoluções sem padding
(Kb=15 → 7 → 5 → 3 → 1; Kb=30 → 22 → 20 → 18 → 16).

`seed_everything` vem imediatamente antes de construir o modelo: é a construção que consome
o RNG global do PyTorch e define os pesos iniciais.

In [ ]:
hsi.seed_everything(config.seed)
model, criterion, optimizer = hsi.build_model(config, data.n_features, data.n_classes, device)

print(model)
print('\nFlatten features:  ', model.n_flat)
print('Parâmetros treináveis:', model.n_parameters)

## Train

O laço percorre até `config.epochs` épocas, registra quatro séries no histórico (`loss`,
`accuracy`, `val_loss`, `val_accuracy`) e para sozinho quando a perda de validação passa 5
épocas sem melhorar. Os pesos finais são os da **última** época, não os da melhor: o critério
de parada olha a melhor, mas não volta atrás.

`fit` repõe a semente do gerador do loader de treino antes da primeira época, então a ordem
dos batches não depende de o loader ter sido iterado antes (para inspecionar um batch, por
exemplo).

In [ ]:
training = hsi.fit(model, loaders, criterion, optimizer, config, device)
print('\n' + training.summary())

## Curvas de treino

In [ ]:
for metric in ('Loss', 'Accuracy'):
    hsi.plots.plot_history(training.history, metric,
                           save_to=OUT_DIR / f'{config.model_name}_{metric}_curve_torch.pdf')
    plt.show()

# Evaluating

## Métricas

`evaluate` faz a predição (`model.eval()` + `no_grad` + softmax explícito, já que a última
camada devolve logits) e calcula tudo de uma vez:

- **OA** (overall accuracy): pesa cada amostra igualmente;
- **AA** (average accuracy): pesa cada *classe* igualmente, por isso cai quando as classes
  minoritárias vão mal;
- **kappa**: desconta o acerto esperado por acaso.

In [ ]:
y_test = data.labels[data.splits.test]
metrics, predicted = hsi.evaluate(model, loaders['test'], criterion, device, y_test)

print(metrics.report)
print(metrics.summary())

### Acurácia por classe

In [ ]:
print(pd.DataFrame({
    'acurácia': metrics.per_class_accuracy,
    'suporte': metrics.confusion.sum(axis=1),
}).rename_axis('classe').to_string(float_format='{:.4f}'.format))

## Matriz de confusão

In [ ]:
hsi.plots.plot_confusion(metrics.confusion)
plt.show()

hsi.plots.plot_confusion(metrics.confusion, normalize=True)
plt.show()

## Leitura binária (reprodução do original)

O notebook original, na seção *"Evaluating (Livro)"*, aplicava a fórmula de TP/TN/FP/FN de um
problema **binário** sobre a matriz de confusão multiclasse: só o bloco 2×2 do canto superior
esquerdo, ignorando todas as outras classes. Mantido aqui para comparação com o trabalho
original — **não é uma métrica válida deste problema**. As métricas corretas são as da célula
anterior (OA, AA, kappa).

In [ ]:
for name, value in hsi.binary_readout(metrics.confusion).items():
    print(f'{name:10s} {value:.4f}')

## Salvar / carregar o modelo

O padrão do PyTorch é salvar apenas o `state_dict` (os tensores de peso) e reconstruir a
arquitetura antes de carregá-lo — por isso o checkpoint guarda também o que `load_checkpoint`
precisa para remontar a rede.

In [ ]:
ckpt_path = hsi.save_checkpoint(model, config, data.n_features, data.n_classes,
                                OUT_DIR / f'{config.model_name}_torch.pt')
print('Modelo salvo em', ckpt_path)

# Recarregar e conferir que a predição bate:
loaded, ckpt = hsi.load_checkpoint(ckpt_path, device)
reloaded = np.argmax(hsi.predict_proba(loaded, loaders['test'], device), axis=1)
print('Predição do modelo recarregado é idêntica:', np.array_equal(reloaded, predicted))

# Baseline × nossa implementação

`hsi.variants` define as duas versões que o trabalho compara:

- **`BASELINE`** — o fluxo da implementação de referência (Souza, 2023): estritamente
  sequencial, com os mesmos hiperparâmetros do notebook original;
- **`OPTIMIZED`** — a nossa: mesmo modelo e mesmos hiperparâmetros, com a preparação
  paralelizada.

Os hiperparâmetros são iguais de propósito. É isso que torna a comparação interpretável:
qualquer diferença de tempo vem da estratégia de execução, e as métricas de qualidade têm de
sair idênticas — se não saírem, a otimização mudou o que o código calcula, e é bug, não ganho.

Para a comparação completa (as duas variantes de ponta a ponta, com tabela de tempos e
verificação das métricas), use o script — ele evita retreinar dentro do notebook:

```bash
python scripts/compare_variants.py -d PU --json outputs/comparacao.json
```

In [ ]:
for variante in hsi.VARIANTS.values():
    print(f'{variante.name} (backend: {variante.backend})')
    print(f'  {variante.description}\n')